In [4]:

import pandas as pd
import geopandas as gpd
import numpy as np
import altair as alt
import os
from pathlib import Path
import eco_style 
alt.themes.enable("light")
import requests
import json
import time
from tqdm import tqdm
import io

In [5]:
# Set relative file paths
script_dir = Path.cwd()
export_path = script_dir.parent / "Data"

In [11]:
# READ UK BUSINESS COUNTS BY 2-DIGIT SIC FROM NOMIS API
# Dataset: NM_141_1 - UK Business Counts (local units by industry and employment size band)
# Geography: United Kingdom (2092957697)

OUTPUT_FILE = export_path / 'ONS/local_units_UKbusinesscounts_long.csv'

BASE_URL = "https://www.nomisweb.co.uk/api/v01/dataset/NM_141_1.data.csv"

# Size band codes in Nomis for NM_141_1
# 0 = Total, 1 = Micro, 2 = Small, 3 = Medium, 4 = Large
SIZE_BAND_CODES = {
    "Total":                    0,
    "Micro (0 to 9)":          10,
    "Small (10 to 49)":        20,
    "Medium-sized (50 to 249)": 30,
    "Large (250+)":             40,
}

INDUSTRY_CODES = (
    "146800641...146800643,146800645...146800673,146800675...146800679,"
    "146800681...146800683,146800685...146800687,146800689...146800693,"
    "146800695,146800696,146800698...146800706,146800708...146800715,"
    "146800717...146800722,146800724...146800728,146800730...146800739"
)

# --- Custom category mapping ---
SECTOR_MAP = {}
def _add(codes, name):
    for c in codes:
        SECTOR_MAP[c] = name

_add([41, 42, 43],                      "Construction")
_add([45, 46],                          "Wholesale Trade")
_add([47],                              "Retail Trade")
_add([55, 56],                          "Hospitality")
_add([62, 63],                          "IT & Computer Services")
_add([69, 70, 71, 72, 73, 74, 75],     "Professional Services")
_add([77, 78, 79, 80, 81, 82],         "Business Support Services")
_add([49, 50, 51, 52, 53],             "Transport & Logistics")
_add([87, 88],                          "Social care")
_add([1],                               "Agriculture")
_add(range(10, 34),                     "Manufacturing")
_add([58, 59, 60, 61],                  "Other Information Services")
_add([35, 36, 37, 38, 39],             "Utilities")
_add([2, 3, 5, 6, 7, 8, 9],           "Other Primary Industries")
_add([90, 91, 92, 93, 94, 95, 96,
      97, 98, 99],                      "Other Services")

def get_sic_code(label):
    try:
        return int(str(label).split(':')[0].strip())
    except (ValueError, AttributeError):
        return None

# --- Fetch data for each size band ---
all_dfs = []

for size_label, size_code in SIZE_BAND_CODES.items():
    print(f"Fetching: {size_label}...")

    params = {
        'geography':            '2092957697',
        'date':                 '2010-2025',
        'industry':             INDUSTRY_CODES,
        'employment_sizeband':  size_code,
        'legal_status':         0,
        'measures':             20100,
        'select':               'DATE_NAME,INDUSTRY_NAME,OBS_VALUE',
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()

    chunk_df = pd.read_csv(io.StringIO(response.text))
    chunk_df['size'] = size_label
    all_dfs.append(chunk_df)

# --- Combine and clean ---
raw = pd.concat(all_dfs, ignore_index=True)
raw.columns = [c.strip().lower() for c in raw.columns]

# Extract SIC code and map to sector
raw['sic'] = raw['industry_name'].apply(get_sic_code)
raw['sector'] = raw['sic'].map(SECTOR_MAP)

# Drop unmapped SICs (e.g. 64, 65, 66, 68, 84, 86)
raw = raw.dropna(subset=['sector'])

# Rename and tidy
raw = raw.rename(columns={'date_name': 'year', 'obs_value': 'site_count'})
raw['year'] = raw['year'].astype(int)
raw['site_count'] = pd.to_numeric(raw['site_count'], errors='coerce').fillna(0).astype(int)

# --- Aggregate to sector level ---
df = (
    raw.groupby(['year', 'size', 'sector'], as_index=False)['site_count']
    .sum()
    .sort_values(['year', 'size', 'sector'])
    .reset_index(drop=True)
)

print(df.head(20))
print(f"\nShape: {df.shape}")
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Fetching: Total...
Fetching: Micro (0 to 9)...
Fetching: Small (10 to 49)...
Fetching: Medium-sized (50 to 249)...
Fetching: Large (250+)...
    year                      size                      sector  site_count
0   2010              Large (250+)                 Agriculture          25
1   2010              Large (250+)   Business Support Services        1185
2   2010              Large (250+)                Construction         310
3   2010              Large (250+)                 Hospitality         185
4   2010              Large (250+)      IT & Computer Services         200
5   2010              Large (250+)               Manufacturing        1425
6   2010              Large (250+)  Other Information Services         310
7   2010              Large (250+)    Other Primary Industries          30
8   2010              Large (250+)              Other Services         295
9   2010              Large (250+)       Professional Services         735
10  2010              Large (250+)

In [8]:
response = requests.get(BASE_URL, params=params)
print(response.url)
print(response.text[:500])

https://www.nomisweb.co.uk/api/v01/dataset/NM_141_1.data.csv?geography=2092957697&date=2010-2025&industry=TYPE602&employment_sizeband=0&legal_status=0&measures=20100&select=DATE_NAME%2CINDUSTRY_NAME%2COBS_VALUE

